# Notebook 03 — Modélisation, MLflow, optimisation

Objectif : comparer des modèles, **tout tracer dans MLflow**, optimiser le modèle et le **seuil métier**, puis **interpréter** (SHAP) et **servir** le modèle.

> **Avant de commencer**, lance MLflow dans un terminal séparé : `uv run mlflow ui` (interface sur http://127.0.0.1:5000).

## 1. Préparer les données et le score métier

In [1]:
# =====================================================================
# Notebook 03 - Modélisation, MLflow, optimisation
# Objectif : comparer des modèles, tout tracer dans MLflow, optimiser le
#            modèle et le seuil, puis interpréter et servir le modèle.
# =====================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow                 # tracking des expériences (cœur MLOps)
import mlflow.sklearn         # log de modèles scikit-learn dans MLflow

# Masquer l'avertissement cosmétique "DataFrame is highly fragmented" de pandas.
# Ce n'est PAS une erreur : il apparaît quand on ajoute des colonnes une à une à
# une table déjà large (agrégations, ratios). Les résultats sont corrects ; on
# masque juste le message pour garder une sortie lisible.
import warnings
from pandas.errors import PerformanceWarning
warnings.simplefilter("ignore", category=PerformanceWarning)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (roc_auc_score, accuracy_score, recall_score,
                             confusion_matrix, make_scorer)

# Recharger le dataset préparé (instantané grâce au format Parquet)
df = pd.read_parquet("../data/processed/dataset_final.parquet")

# Séparer la cible (y) des features (X) ; SK_ID_CURR est un identifiant, pas un signal
y = df["TARGET"]                                  # la cible : 0 = remboursé, 1 = défaut
X = df.drop(columns=["TARGET", "SK_ID_CURR"])     # les variables explicatives

# Sécurité : remplacer d'éventuels infinis (divisions par zéro dans les ratios)
# par NaN, pour que l'imputation fonctionne (sinon "Input X contains infinity").
X = X.replace([np.inf, -np.inf], np.nan)

# Filet de sécurité (dtypes) : ne conserver que les colonnes numériques et
# booléennes, puis tout convertir en float32. Cela (a) écarte d'éventuelles
# colonnes texte non encodées qui forceraient un tableau "object" (très
# gourmand en RAM -> MemoryError), (b) réduit fortement l'empreinte mémoire.
X = X.select_dtypes(include=["number", "bool"]).astype("float32")
print("X prêt :", X.shape, "| dtypes :", set(map(str, X.dtypes.unique())))

# Découpage stratifié : conserve la proportion 92/8 dans train ET validation
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print("Train :", X_train.shape, "| Valid :", X_valid.shape)

X prêt : (307507, 635) | dtypes : {'float32'}
Train : (246005, 635) | Valid : (61502, 635)


In [2]:
# --- Score métier : un faux négatif (FN) coûte 10x un faux positif (FP) ---
def cout_metier(y_true, y_pred, cout_fn=10, cout_fp=1):
    """Coût total des erreurs. Un FN (mauvais client accepté) = 10, un FP = 1.

    Args:
        y_true: vraies classes (0/1).
        y_pred: classes prédites (0/1).
        cout_fn (int): coût d'un faux négatif.
        cout_fp (int): coût d'un faux positif.

    Returns:
        int: coût total pondéré des erreurs (plus c'est bas, mieux c'est).
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return cout_fn * fn + cout_fp * fp


def score_metier_normalise(y_true, y_pred, cout_fn=10, cout_fp=1):
    """Coût moyen par client (permet de comparer des jeux de tailles différentes)."""
    return cout_metier(y_true, y_pred, cout_fn, cout_fp) / len(y_true)

## 2. Configurer MLflow

In [3]:
# Où MLflow stocke les runs (dossier local mlruns/) et nom de l'expérience
mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("scoring_credit_pretadepenser")

<Experiment: artifact_location='file:../mlruns/1', experiment_id='1', lifecycle_stage='active', name='scoring_credit_pretadepenser', tags={}>

## 3. Fonction d'évaluation réutilisable

Entraîne un modèle en validation croisée, calcule les métriques et crée un **run MLflow complet** (params, métriques, tags, modèle).

In [4]:
# Validation croisée stratifiée (conserve la proportion des classes dans chaque pli)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Notre coût est à MINIMISER -> greater_is_better=False (scikit-learn le stocke en négatif)
scorer_cout = make_scorer(score_metier_normalise, greater_is_better=False)
scoring = {"AUC": "roc_auc", "cout_metier": scorer_cout}


def evaluer_modele(nom, modele, X_tr, y_tr, X_va, y_va, tags=None):
    """Entraîne un modèle en CV, logge tout dans MLflow et renvoie le modèle + probas.

    Args:
        nom (str): nom du run (et du modèle).
        modele: estimateur scikit-learn (ou pipeline).
        X_tr, y_tr: données d'entraînement.
        X_va, y_va: données de validation.
        tags (dict): annotations supplémentaires pour le run MLflow.

    Returns:
        tuple: (modèle entraîné, probabilités prédites sur la validation).
    """
    with mlflow.start_run(run_name=nom):
        # Annotations (toujours annoter les runs pour s'y retrouver dans l'UI)
        mlflow.set_tag("modele", nom)
        mlflow.set_tag("auteur", "Yohan Parent")
        for k, v in (tags or {}).items():
            mlflow.set_tag(k, v)

        # Validation croisée stratifiée (robustesse de l'évaluation)
        res = cross_validate(modele, X_tr, y_tr, cv=cv, scoring=scoring,
                             return_train_score=True, n_jobs=1)
        auc_cv  = res["test_AUC"].mean()
        cout_cv = -res["test_cout_metier"].mean()   # on remet le signe positif
        auc_train = res["train_AUC"].mean()

        # Entraînement final + métriques sur la validation (seuil 0.5 par défaut)
        modele.fit(X_tr, y_tr)
        proba = modele.predict_proba(X_va)[:, 1]
        preds = (proba >= 0.5).astype(int)

        # Log des métriques dans MLflow
        mlflow.log_metric("AUC_cv", auc_cv)
        mlflow.log_metric("AUC_train", auc_train)        # pour détecter l'overfitting
        mlflow.log_metric("cout_metier_cv", cout_cv)
        mlflow.log_metric("AUC_valid", roc_auc_score(y_va, proba))
        mlflow.log_metric("accuracy_valid", accuracy_score(y_va, preds))
        mlflow.log_metric("recall_valid", recall_score(y_va, preds))

        # Log du modèle entraîné (artefact réutilisable)
        mlflow.sklearn.log_model(modele, "model")
        print(f"{nom:18s} | AUC_cv={auc_cv:.3f} | AUC_train={auc_train:.3f} | coût={cout_cv:.4f}")
        return modele, proba

## 4. Étape 3 — Comparer plusieurs modèles (avec gestion du déséquilibre)

Toutes les familles gèrent le déséquilibre par pondération des classes (`class_weight` / `scale_pos_weight`).

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# ratio négatifs/positifs (~11) pour pondérer XGBoost
ratio = (y_train == 0).sum() / (y_train == 1).sum()

# La régression logistique a besoin d'imputation + standardisation :
# on l'emballe dans un petit pipeline (évite la fuite de données en CV).
logreg = SkPipeline([
    ("imputer", SimpleImputer(strategy="median")),   # remplace les NaN par la médiane
    ("scaler", StandardScaler()),                     # met les variables à la même échelle
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000)),
])

# Dictionnaire des modèles à comparer
modeles = {
    "logreg_baseline": logreg,
    "random_forest":   RandomForestClassifier(class_weight="balanced",
                          n_estimators=200, n_jobs=-1, random_state=42),
    "lightgbm":        LGBMClassifier(class_weight="balanced",
                          random_state=42, verbose=-1),
    "xgboost":         XGBClassifier(scale_pos_weight=ratio, eval_metric="auc",
                          random_state=42, n_jobs=-1),
}

# Entraîner et logguer chaque modèle dans MLflow
resultats = {}
for nom, modele in modeles.items():
    m, proba = evaluer_modele(nom, modele, X_train, y_train, X_valid, y_valid,
                              tags={"etape": "comparaison_modeles"})
    resultats[nom] = (m, proba)

2026/06/29 20:09:28 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Shadow\AppData\Local\Temp\tmpz1a255gj\model\model.pkl, flavor: sklearn), fall back to return ['scikit-learn==1.9.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback.
2026/06/29 20:09:28 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


logreg_baseline    | AUC_cv=0.768 | AUC_train=0.778 | coût=0.5137


2026/06/29 20:16:06 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


random_forest      | AUC_cv=0.752 | AUC_train=1.000 | coût=0.7815


2026/06/29 20:17:47 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


lightgbm           | AUC_cv=0.780 | AUC_train=0.836 | coût=0.4972


2026/06/29 20:19:46 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


xgboost            | AUC_cv=0.761 | AUC_train=0.914 | coût=0.5307


### Variante SMOTE (sur-échantillonnage, dans un pipeline pour éviter la fuite de données)

In [6]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# SMOTE doit s'appliquer SEULEMENT au train de chaque pli -> on l'enferme dans un pipeline
pipeline_smote = ImbPipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("smote", SMOTE(random_state=42)),     # crée des exemples synthétiques de la classe rare
    ("clf", LGBMClassifier(random_state=42, verbose=-1)),
])
evaluer_modele("lightgbm_smote", pipeline_smote, X_train, y_train, X_valid, y_valid,
               tags={"desequilibre": "SMOTE"})

C:\Users\Shadow\Documents\OC\PROJET 6\projet6-mlops\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Shadow\Documents\OC\PROJET 6\projet6-mlops\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Shadow\Documents\OC\PROJET 6\projet6-mlops\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Shadow\Documents\OC\PROJET 6\projet6-mlops\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\Shadow\Documents\OC\PROJET 6\projet6-mlops\.venv\Lib\site-packages\sklearn\utils\valida

lightgbm_smote     | AUC_cv=0.773 | AUC_train=0.808 | coût=0.7905


(Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                 ('smote', SMOTE(random_state=42)),
                 ('clf', LGBMClassifier(random_state=42, verbose=-1))]),
 array([0.04681058, 0.07159225, 0.0994197 , ..., 0.04140464, 0.09864042,
        0.19210154], shape=(61502,)))

> **Pense à prendre tes screenshots de l'UI MLflow** (tableau des runs + vue « Compare »).

**Sauvegarde Git :**
```powershell
git add notebooks/03_modelisation_mlflow.ipynb
git commit -m "Etape 3 : comparaison de 4 modeles + gestion du desequilibre (MLflow)"
git push
```

## 5. Étape 4 — Optimiser les hyperparamètres (Optuna)

In [ ]:
import optuna
from sklearn.model_selection import cross_val_score
optuna.logging.set_verbosity(optuna.logging.WARNING)   # silence les logs verbeux


def objectif(trial):
    """Fonction objectif d'Optuna : renvoie l'AUC CV moyen pour un jeu d'hyperparamètres.

    Optuna propose une combinaison, on l'évalue en validation croisée, et il
    apprend des essais passés pour cibler les zones prometteuses.
    """
    params = {
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 5.0),       # régularisation
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 5.0),     # régularisation
    }
    modele = LGBMClassifier(class_weight="balanced", random_state=42,
                            verbose=-1, **params)
    # On optimise l'AUC en validation croisée
    return cross_val_score(modele, X_train, y_train, cv=cv,
                           scoring="roc_auc", n_jobs=1).mean()


# Lancer l'étude d'optimisation (augmente n_trials si tu as le temps)
etude = optuna.create_study(direction="maximize")
etude.optimize(objectif, n_trials=30)
print("Meilleur AUC CV :", round(etude.best_value, 4))
print("Meilleurs params :", etude.best_params)

## 6. Optimiser le seuil de décision sur le coût métier

Le seuil par défaut (0,5) n'est pas optimal : comme un FN coûte 10x un FP, on a intérêt à abaisser le seuil.

In [ ]:
# Modèle final avec les meilleurs hyperparamètres trouvés par Optuna
best_model = LGBMClassifier(class_weight="balanced", random_state=42,
                            verbose=-1, **etude.best_params)
best_model.fit(X_train, y_train)
proba_valid = best_model.predict_proba(X_valid)[:, 1]   # probabilités de défaut sur la validation

# Tester tous les seuils de 0.05 à 0.95 et calculer le coût métier pour chacun
seuils = np.arange(0.05, 0.96, 0.01)
couts = np.array([cout_metier(y_valid, (proba_valid >= s).astype(int)) for s in seuils])

seuil_optimal = seuils[couts.argmin()]      # le seuil qui minimise le coût
cout_optimal = couts.min()
cout_defaut = cout_metier(y_valid, (proba_valid >= 0.5).astype(int))
print(f"Seuil optimal = {seuil_optimal:.2f}")
print(f"Coût au seuil optimal = {cout_optimal}  vs  coût à 0.5 = {cout_defaut}")
print(f"Gain = {100 * (cout_defaut - cout_optimal) / cout_defaut:.1f} %")

### Courbe coût métier vs seuil (exigée par la grille)

In [ ]:
# Tracer le coût métier en fonction du seuil, et marquer le seuil optimal
plt.figure(figsize=(8, 5))
plt.plot(seuils, couts, label="Coût métier")
plt.axvline(seuil_optimal, color="red", linestyle="--",
            label=f"Seuil optimal = {seuil_optimal:.2f}")
plt.axvline(0.5, color="grey", linestyle=":", label="Seuil par défaut = 0.5")
plt.xlabel("Seuil de décision"); plt.ylabel("Coût métier total")
plt.title("Coût métier en fonction du seuil"); plt.legend()
plt.tight_layout()
plt.savefig("cout_vs_seuil.png", dpi=120)   # sauvegarde pour la loguer dans MLflow
plt.show()

## 7. Run final + enregistrement au Model Registry

In [ ]:
# Run final : on logge les hyperparamètres optimaux, le seuil, les métriques et la courbe,
# et on enregistre le modèle dans le Model Registry (versionné).
with mlflow.start_run(run_name="lightgbm_final_optimise"):
    mlflow.set_tag("statut", "modele_final")
    mlflow.log_params(etude.best_params)                  # tous les hyperparamètres optimaux
    mlflow.log_param("seuil_optimal", float(seuil_optimal))

    preds_opt = (proba_valid >= seuil_optimal).astype(int)
    mlflow.log_metric("AUC_valid", roc_auc_score(y_valid, proba_valid))
    mlflow.log_metric("cout_metier_optimal", float(cout_optimal))
    mlflow.log_metric("recall_defaut", recall_score(y_valid, preds_opt))

    mlflow.log_artifact("cout_vs_seuil.png")              # on attache la courbe au run
    # registered_model_name => crée/incrémente une version dans le Model Registry
    mlflow.sklearn.log_model(best_model, "model",
                             registered_model_name="scoring_credit_model")
print("Run final enregistré et modèle versionné dans le registry.")

**Sauvegarde Git :**
```powershell
git add notebooks/03_modelisation_mlflow.ipynb
git commit -m "Etape 4 : optimisation Optuna + seuil metier + enregistrement au registry"
git push
```

## 8. Interprétabilité : feature importance globale et locale (SHAP)

In [ ]:
import shap

# Explainer optimisé pour les modèles à base d'arbres (rapide et exact)
explainer = shap.TreeExplainer(best_model)
X_sample = X_valid.sample(2000, random_state=42)    # échantillon pour la rapidité
shap_values = explainer.shap_values(X_sample)
# Pour un classifieur binaire, shap_values peut être une liste : on prend la classe 1 (défaut)
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

In [ ]:
# --- Importance GLOBALE : quelles variables comptent le plus, tous clients confondus ---
shap.summary_plot(sv, X_sample, plot_type="bar", show=False)
plt.tight_layout(); plt.savefig("shap_global.png", dpi=120, bbox_inches="tight"); plt.show()

In [ ]:
# --- Importance LOCALE : pourquoi CE client a CE score (pour un chargé d'études) ---
i = 0   # le i-ème client de l'échantillon
shap.plots.waterfall(shap.Explanation(
    values=sv[i],
    base_values=explainer.expected_value if np.isscalar(explainer.expected_value)
                else explainer.expected_value[1],
    data=X_sample.iloc[i],
    feature_names=X_sample.columns.tolist()), show=False)
plt.tight_layout(); plt.savefig("shap_local.png", dpi=120, bbox_inches="tight"); plt.show()

In [ ]:
# Conserver les figures SHAP dans MLflow
with mlflow.start_run(run_name="interpretabilite_shap"):
    mlflow.log_artifact("shap_global.png")
    mlflow.log_artifact("shap_local.png")

**Sauvegarde Git :**
```powershell
git add notebooks/03_modelisation_mlflow.ipynb
git commit -m "Interpretabilite : feature importance globale et locale (SHAP)"
git push
```

## 9. Tester le serving MLflow (pré-production)

Lance d'abord le serveur dans un **terminal séparé** :

```powershell
uv run mlflow models serve -m "models:/scoring_credit_model/1" -p 1234 --env-manager=local
```

Puis exécute la cellule ci-dessous pour interroger l'API.

In [ ]:
# --- Tester le serving : envoyer 3 clients à l'API et récupérer les prédictions ---
import requests   # envoyer des requêtes HTTP
import json       # formater les données en JSON

# 3 clients de validation ; fillna(0) bouche les NaN pour que l'envoi JSON soit valide
exemple = X_valid.iloc[:3].fillna(0)

# Format attendu par l'API MLflow : "dataframe_split" = colonnes + données
payload = {"dataframe_split": {
    "columns": exemple.columns.tolist(),
    "data": exemple.values.tolist()
}}

# Appel POST sur l'endpoint /invocations du serveur lancé sur le port 1234
reponse = requests.post(
    "http://127.0.0.1:1234/invocations",
    headers={"Content-Type": "application/json"},
    data=json.dumps(payload),
)
print(reponse.json())   # les prédictions renvoyées par l'API

## 10. Synthèse métier (à adapter avec tes chiffres réels)

> Le modèle LightGBM optimisé atteint un AUC de **0,78** (bon pouvoir de classement). Au seuil par défaut 0,5, le coût métier est de X. En abaissant le seuil à **0,XX**, on accepte davantage de faux positifs (peu coûteux) pour éviter des faux négatifs (10× plus coûteux), faisant baisser le coût de **−ZZ %**. Le recall sur la classe « défaut » passe de … à … . Ce réglage traduit la priorité métier : limiter les pertes en capital.

**Sauvegarde Git finale (notebook + captures d'écran) :**
```powershell
git add notebooks/03_modelisation_mlflow.ipynb screenshots/
git commit -m "Test du serving MLflow + captures d'ecran des livrables"
git push
```